<a href="https://colab.research.google.com/github/technologymalviya/MachineLeaning/blob/main/Gita.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bhagavad Gita QA Model

Train a retrieval-based question-answering model on the **Srimad Bhagavad Gita** (700 verses) using Hugging Face datasets. The trained model is saved to `gita_model/` and can answer Gita-related queries.

In [ ]:
# Install required packages (run once in Colab/local)
%pip install -q datasets scikit-learn joblib pandas numpy

In [ ]:
import json
import pandas as pd
from datasets import load_dataset
from gita_qa_model import GitaQAModel

## 1. Load Gita Datasets

- **Bhagavad-Gita_Dataset**: 700 verses in Sanskrit, Hindi, and English
- **Bhagavad-Gita-QA**: 3,500 verse-aligned question-answer pairs for training and evaluation

In [ ]:
verses_ds = load_dataset("JDhruv14/Bhagavad-Gita_Dataset", split="train")
qa_ds = load_dataset("JDhruv14/Bhagavad-Gita-QA", split="train")

verses_df = verses_ds.to_pandas()
qa_df = qa_ds.to_pandas()

print(f"Verses: {len(verses_df)} | QA pairs: {len(qa_df)}")
print("\nSample verse:")
print(verses_df.iloc[0][['chapter', 'verse', 'english']])
print("\nSample QA:")
print(qa_df.iloc[0][['chapter_no', 'verse_no', 'question', 'answer']])

## 2. Train the Model

Hybrid retrieval model (`gita_qa_model.py`):
1. **TF-IDF verse retrieval** — finds the most relevant Gita verse(s) for a question
2. **QA matching** — picks the best answer from training Q&A pairs for the retrieved verse

In [ ]:
model = GitaQAModel()
model.fit(verses_df, qa_df)
print(f"Training complete — {len(model.train_qa)} QA pairs used for training")

## 3. Test & Accuracy Report

Evaluation on a held-out **20% test set** (700 questions):

| Metric | Description |
|--------|-------------|
| **Verse Top-1 Accuracy** | Correct chapter.verse retrieved as #1 match |
| **Verse Top-3 Accuracy** | Correct verse found in top 3 results |
| **Answer Token F1** | Word overlap between predicted and expected answer |
| **Answer Cosine Similarity** | Semantic similarity of answers (TF-IDF) |
| **Overall Knowledge Score** | Weighted composite (40% verse + 60% answer) |

In [ ]:
metrics = model.evaluate()

print("=" * 50)
print("  BHAGAVAD GITA MODEL — ACCURACY REPORT")
print("=" * 50)
for key, value in metrics.items():
    label = key.replace('_', ' ').title()
    suffix = '%' if 'accuracy' in key or 'f1' in key or 'similarity' in key or 'score' in key else ''
    print(f"  {label:<30} {value}{suffix}")
print("=" * 50)

## 4. Save Trained Model

The model is saved to `gita_model/gita_qa_model.joblib` for reuse in other scripts or applications.

In [ ]:
model.save('gita_model')

## 5. Try Gita Queries

Ask questions about the Bhagavad Gita — the model returns an answer with relevant verse references.

In [ ]:
def print_gita_answer(result):
    print(f"Question: {result['question']}\n")
    print(f"Answer: {result['answer']}\n")
    print("Relevant Verses:")
    for v in result['verses']:
        print(f"  Chapter {v['chapter']}, Verse {v['verse']} (score: {v['score']:.3f})")
        print(f"    EN: {v['english'][:120]}...")
    print("-" * 60)

sample_questions = [
    "What does Krishna say about karma yoga?",
    "What is the nature of the soul according to the Gita?",
    "Why was Arjuna reluctant to fight?",
    "What is dharma?",
]

for q in sample_questions:
    print_gita_answer(model.ask(q))

## 6. Load Saved Model (for future use)

Use this cell in a new session to load the pre-trained model without retraining.

In [ ]:
loaded_model = GitaQAModel.load('gita_model')

my_question = "What is the teaching on detachment?"
result = loaded_model.ask(my_question)
print_gita_answer(result)